# Lesson 30: Inference in MLR Models

Two datasets:

- **Part 1: Snowy plover.** Bootstrap confidence intervals for regression coefficients, and see how collinearity gets in the way.
- **Part 2: Hybrid cars.** Bootstrap measures of model fit (R², adjusted R², and the gain from adding features) to decide whether a model, or a group of features, really helps.

Section names match the **Jupyter demo** tags on the slides.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.linear_model as lm

# big font helper
def adjust_fontsize(size=None):
    SMALL_SIZE = 8
    MEDIUM_SIZE = 10
    BIGGER_SIZE = 12
    if size != None:
        SMALL_SIZE = MEDIUM_SIZE = BIGGER_SIZE = size

    plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
    plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
    plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
    plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
    plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
    plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
    plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

plt.style.use('fivethirtyeight')
sns.set_context("talk")
sns.set_theme()
#plt.style.use('default') # revert style to default mpl
adjust_fontsize(size=20)
%matplotlib inline

---

# Part 1: Snowy plover

This example is from the textbook: [link](http://www.textbook.ds100.org/ch/19/mult_inference.html#).

## The Data ##
The [Snowy Plover](https://www.audubon.org/field-guide/bird/snowy-plover) is a tiny bird that lives on the coast in parts of California and elsewhere. It is so small that it is vulnerable to many predators and to people and dogs that don't look where they are stepping when they go to the beach. It is considered endangered in many parts of the US.

The data are about the eggs and newly-hatched chicks of the Snowy Plover.

The data were collected at the Point Reyes National Seashore.  The goal was to see how the size of an egg could be used to predict the weight of the resulting chick. The bigger the newly-hatched chick, the more likely it is to survive.

Each row of the data frame below corresponds to one Snowy Plover egg and the resulting chick. Note how tiny the bird is:

* Egg Length and Egg Breadth (widest diameter) are measured in millimeters
* Egg Weight and Bird Weight are measured in grams; for comparison, a standard paper clip weighs about one gram

In [ ]:
eggs = pd.read_csv('data/snowy_plover.csv')
eggs.head()

In [ ]:
eggs.shape

For a particular egg, $x$ is the vector of weight, length, and breadth. The model is

$$
f_\theta(x) ~ = ~ \theta_0 + \theta_1\text{egg_weight} + \theta_2\text{egg_length} + \theta_3\text{egg_breadth} + \epsilon
$$

- For each $i$, the parameter $\theta_i$ is a fixed number but it is unobservable. We can only estimate it.
- The random error $\epsilon$ is also unobservable, but it is assumed to have expectation 0 and be independent and identically distributed across eggs.

In [ ]:
y = eggs["bird_weight"]
X = eggs[["egg_weight", "egg_length", "egg_breadth"]]

model = lm.LinearRegression(fit_intercept=True).fit(X, y)

display(pd.DataFrame([model.intercept_] + list(model.coef_),
             columns=['theta_hat'],
             index=['intercept', 'egg_weight', 'egg_length', 'egg_breadth']))

print("RMSE", np.sqrt(np.mean((y - model.predict(X)) ** 2)))

## Snowy plover: bootstrapping coefficients

Let's try bootstrapping the sample to obtain a 95% confidence interval for the parameter $\theta_1$ corresponding to egg weight.

This code uses `df.sample` ([link](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html)) to generate a bootstrap sample of the same size of the original sample.

In [ ]:
def get_param1(model):
    # first feature (egg_weight)
    return model.coef_[0]

def bootstrap_params(sample_df, get_param_fn=get_param1, n_iters=10000):
    """
    sample_df: the bootstrap population
    """
    n = len(sample_df)
    estimates = []
    for i in range(n_iters):
        # resample n times with replacement
        # i.e., get a new sample of same size
        # using df.sample(...)
        resample = sample_df.sample(n, replace=True)

        # train model with this bootstrap sample
        resample_y = resample["bird_weight"]
        resample_X = resample[["egg_weight", "egg_length", "egg_breadth"]]
        model = lm.LinearRegression()
        model.fit(resample_X, resample_y)

        # include the estimate
        estimate = get_param_fn(model)
        estimates.append(estimate)
    lower = np.percentile(estimates, 2.5, axis=0)
    upper = np.percentile(estimates, 97.5, axis=0)
    conf_interval = (lower, upper)
    return conf_interval

In [ ]:
approx_conf1 = bootstrap_params(eggs, get_param1)
approx_conf1

Because the 95% confidence interval includes 0, we cannot reject the null hypothesis that the true parameter $\theta_1$ is 0.

<br/>

### Testing all the coefficients

Let's bootstrap again and compute 95% confidence intervals for all 4 parameters, including the intercept term:

In [ ]:
def get_all_params(model):
    # all features
    return [model.intercept_] + list(model.coef_)

approx_confs = bootstrap_params(eggs, get_param_fn=get_all_params)
egg_ci = pd.DataFrame(np.array(approx_confs).T, columns=['lower', 'upper'],
                      index=['theta_intercept', 'theta_egg_weight', 'theta_egg_length', 'theta_egg_breadth'])
egg_ci

We find something strange: all of the confidence intervals for the slopes contain 0, which prevents us from concluding that *any* variable is significantly related to the response.

<br/><br/>

---

## Collinearity

To see what's going on, we'll make a scatter plot matrix for the data.

In [ ]:
sns.pairplot(eggs, height=1.8);

This shows that `bird_weight`
is highly correlated with all the other
variables (the bottom row), which means fitting a linear model is a good idea.
But we also see that `egg_weight` is highly correlated with all the variables
(the top row).
This means we can't increase one covariate while
keeping the others constant. The individual slopes have no meaning.

Here's the correlations showing this more succinctly:

In [ ]:
eggs.corr().round(2)

<br/><br/>

---

## Changing Our Modeling Features

One way to fix this is to fit a model that only uses `egg_weight`.
This model performs almost as well as the model that uses all three variables,
and the confidence interval for $\theta_1$ doesn't
contain zero.

In [ ]:
y = eggs["bird_weight"]
X = eggs[["egg_weight"]]

model = lm.LinearRegression(fit_intercept=True).fit(X, y)

display(pd.DataFrame([model.intercept_] + list(model.coef_),
             columns=['theta_hat'],
             index=['intercept', 'egg_weight']))
print("RMSE", np.sqrt(np.mean((y - model.predict(X)) ** 2)))

In [ ]:
def bootstrap_egg_weight_only(sample_df, n_iters=10000):
    """
    copied over for convenience
    """
    n = len(sample_df)
    estimates = []
    for i in range(n_iters):
        resample = sample_df.sample(n, replace=True)

        resample_y = resample["bird_weight"]
        resample_X = resample[["egg_weight"]] # just one feature + intercept
        model = lm.LinearRegression()
        model.fit(resample_X, resample_y)
        estimates.append(model.coef_[0])
    lower = np.percentile(estimates, 2.5, axis=0)
    upper = np.percentile(estimates, 97.5, axis=0)
    conf_interval = (lower, upper)
    return conf_interval

approx_conf_egg_weight_only = bootstrap_egg_weight_only(eggs)
approx_conf_egg_weight_only

It's no surprise that if you want to predict the
weight of the newly-hatched chick, using the weight
of the egg is your best move.

As this example shows, checking for collinearity is important for inference.
When we fit a model on highly correlated variables, we might not be able to
use confidence intervals to conclude that variables are
related to the prediction.

**Preview of Part 2:** the three egg measurements together clearly *do* predict chick weight, even though no single coefficient's CI excludes 0. In Part 2 we'll bootstrap the model's **adjusted R²** instead. Come back and try it here at the end:

In [ ]:
def adjusted_r2(data, features, response):
    X, y = data[features], data[response]
    y_hat = lm.LinearRegression().fit(X, y).predict(X)
    r2 = 1 - np.sum((y - y_hat) ** 2) / np.sum((y - y.mean()) ** 2)
    n, p = len(y), len(features)
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

egg_features = ["egg_weight", "egg_length", "egg_breadth"]
boot_egg_adj = [adjusted_r2(eggs.sample(len(eggs), replace=True), egg_features, "bird_weight")
                for _ in range(2000)]
print("adjusted R^2:", round(adjusted_r2(eggs, egg_features, "bird_weight"), 3),
      "  95% CI:", np.percentile(boot_egg_adj, [2.5, 97.5]).round(3))

---

# Part 2: Hybrid cars

## Hybrid cars: recap

Same data and models as Lessons 26–27b: acceleration, mpg, their transformations, and one-hot encoded car class (drop-first, so Compact is the baseline).

In [ ]:
from sklearn.preprocessing import OneHotEncoder

rng = np.random.default_rng(3022)   # fixed seed so results match the slides

hybrid = pd.read_csv('data/hybrid.csv')
n = len(hybrid)

hybrid['acc_sq'] = hybrid['acceleration'] ** 2
hybrid['mpg_sqrt'] = np.sqrt(hybrid['mpg'])
hybrid['noise'] = rng.normal(size=n)          # a useless feature for the last section

ohe = OneHotEncoder(drop='first')
ohe.fit(hybrid[['class']])
class_names = list(ohe.get_feature_names_out())
cars = hybrid.join(pd.DataFrame(ohe.transform(hybrid[['class']]).toarray(),
                                columns=class_names, index=hybrid.index))

quant = ['acceleration', 'mpg', 'acc_sq', 'mpg_sqrt']
models = {
    'SLR':    ['acceleration'],
    'MLR V1': ['acceleration', 'mpg'],
    'MLR V2': quant,
    'MLR V3': quant + class_names,
}

In [ ]:
def fit_stats(data, features):
    """Fit least squares (with intercept) on data; return SSE, R^2 and adjusted R^2."""
    X, y = data[features], data['msrp']
    y_hat = lm.LinearRegression().fit(X, y).predict(X)
    sse = np.sum((y - y_hat) ** 2)
    r2 = 1 - sse / np.sum((y - y.mean()) ** 2)
    p, n_rows = len(features), len(y)
    adj_r2 = 1 - (1 - r2) * (n_rows - 1) / (n_rows - p - 1)
    return {'SSE': sse, 'R2': r2, 'adj_R2': adj_r2}

summary = pd.DataFrame({name: fit_stats(cars, f) for name, f in models.items()}).T
summary['RMSE'] = np.sqrt(summary['SSE'] / n)
summary[['RMSE', 'R2', 'adj_R2']].round(3)

Every added feature raises R². Is each improvement more than chance?

We'll bootstrap the same way as for the snowy plover, but the statistic can be anything we compute from a fitted model: a coefficient, R², adjusted R², or the gain from adding features. This helper resamples the rows and returns the statistic for each resample.

With only 4 minivans, an occasional resample has no car from some class. We skip those resamples, since the class models can't be fit on them.

In [ ]:
def bootstrap(statistic, data, B=2000):
    """Resample the rows of data B times; return statistic(resample) for each one."""
    results = []
    while len(results) < B:
        resample = data.sample(len(data), replace=True, random_state=rng)
        if resample['class'].nunique() < data['class'].nunique():
            continue                                   # a class is missing: skip this resample
        results.append(statistic(resample))
    return np.array(results)

def ci95(values):
    return np.percentile(values, [2.5, 97.5], axis=0)

## Class coefficients

Which classes differ, holding acceleration and mpg fixed? Bootstrap the 6 class coefficients in MLR V3. Each compares a class to Compact cars **with the same acceleration and mpg**.

In [ ]:
full_features = quant + class_names
coef_obs = lm.LinearRegression().fit(cars[full_features], cars['msrp']).coef_[len(quant):]
boot_coefs = bootstrap(lambda d: lm.LinearRegression().fit(d[full_features], d['msrp']).coef_[len(quant):], cars)
lo, hi = ci95(boot_coefs)
coef_ci = pd.DataFrame({'coefficient': coef_obs, 'CI low': lo, 'CI high': hi}, index=class_names).round(0)
coef_ci['CI excludes 0'] = (coef_ci['CI low'] > 0) | (coef_ci['CI high'] < 0)
coef_ci

Three classes (Large, Pickup Truck, Two Seater) have intervals that exclude 0. With 6 intervals at once, keep in mind that one could exclude 0 by chance.

## Bootstrap R²

How much would R² change with a different sample of cars?

In [ ]:
def r2_and_adj(data):
    s = fit_stats(data, quant)
    return [s['R2'], s['adj_R2']]

boot_v2 = bootstrap(r2_and_adj, cars)
print('R^2 (MLR V2):         ', summary.loc['MLR V2', 'R2'].round(3), '  95% CI:', ci95(boot_v2[:, 0]).round(3))
print('adjusted R^2 (MLR V2):', summary.loc['MLR V2', 'adj_R2'].round(3), '  95% CI:', ci95(boot_v2[:, 1]).round(3))

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(boot_v2[:, 1], bins=40, edgecolor='white')
lo, hi = ci95(boot_v2[:, 1])
plt.axvline(lo, color='goldenrod'); plt.axvline(hi, color='goldenrod')
plt.axvline(0, color='k', ls='--', label='0: no better than the mean')
plt.xlim(-0.05, 1); plt.xlabel('bootstrapped adjusted R²'); plt.legend();

**Does the model beat the mean?** Use **adjusted R²**, not R². Even useless features give R² > 0 in a sample, but adjusted R² is about 0 on average when the features don't help. The 95% CI for adjusted R² is far above 0, so we have strong evidence that MLR V2 beats the constant model.

## Bootstrap the improvement

Reduced model: MLR V2 (4 quantitative features). Full model: MLR V3 (the same 4 + the 6 class columns). They're **nested**.

In each resample, fit both models and record how much adjusted R² goes up.

In [ ]:
def adj_gain(data, reduced, full):
    return fit_stats(data, full)['adj_R2'] - fit_stats(data, reduced)['adj_R2']

gain_obs = adj_gain(cars, quant, quant + class_names)
boot_gain = bootstrap(lambda d: adj_gain(d, quant, quant + class_names), cars)
print('gain in adjusted R^2:', gain_obs.round(3), '  95% CI:', ci95(boot_gain).round(3))

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(boot_gain, bins=40, edgecolor='white')
lo, hi = ci95(boot_gain)
plt.axvline(lo, color='goldenrod'); plt.axvline(hi, color='goldenrod')
plt.axvline(0, color='k', ls='--', label='0: no improvement')
plt.xlabel('bootstrapped gain in adjusted R² from adding class'); plt.legend();

The whole interval is above 0: strong evidence that car class explains MSRP beyond acceleration and mpg.

## Noise feature

Add a column of random noise to MLR V2. It can't really help, so a good method should say "no".

In [ ]:
def gains(data):
    red, full = fit_stats(data, quant), fit_stats(data, quant + ['noise'])
    noise_coef = lm.LinearRegression().fit(data[quant + ['noise']], data['msrp']).coef_[-1]
    return [full['R2'] - red['R2'], full['adj_R2'] - red['adj_R2'], noise_coef]

obs = gains(cars)
boot_noise = bootstrap(gains, cars)
for k, name in enumerate(['gain in R^2', 'gain in adjusted R^2', 'noise coefficient']):
    print(f'{name:22s} observed: {obs[k]:10.4f}   95% CI: {ci95(boot_noise[:, k]).round(4)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, k, label in zip(axes, [0, 1], ['gain in R²', 'gain in adjusted R²']):
    ax.hist(boot_noise[:, k], bins=40, edgecolor='white')
    ax.axvline(0, color='k', ls='--')
    ax.set_xlabel(f'bootstrapped {label} from adding noise')

- The **gain in R²** can never be negative (R² never goes down when a feature is added), so its CI can never include values below 0. It can't tell us "no".
- The **gain in adjusted R²** and the **noise coefficient** both have CIs that contain 0: no evidence the noise helps.

**Takeaway:** to decide whether features help, bootstrap the gain in adjusted R² (for a group of features) or a coefficient (for one feature), never the gain in R².

Now go back to the end of Part 1 and bootstrap the snowy plover model's adjusted R².